# Assessment Notebook

In [1]:
# Numerical arrays
import numpy as np

## Problem 1: Representing SHA-256 Data

Problem 1 involves investigating how different types of data used in the SHA-256 algorithm can be represented in Python. This includes 32-bit words, sequences of words, input message, 512-bit message blocks, the final 256-bit hash value, and big-endian integers. Small Python examples will be used to demonstrate these representations.

### 32-bit Words

In [2]:
word = np.uint32(42)

print(word)
print(word.dtype)
print(f"{word:032b}")

42
uint32
00000000000000000000000000101010


32-bit words are used by SHA-256 when performing calculations. A 32-bit word consists of 32 binary digits and can represent values ranging from 0 to 4,294,967,295.

In Python, NumPy provides the `np.uint32` data type, which can be used to represent these values. Unlike Python's ordinary integer type that can grow beyond 32 bits, `np.uint32` provides a fixed-width unsigned 32-bit integer making it suitable for SHA-256.

The example above creates a 32-bit word containing the value 42 and displays its data type and binary representation. Other numbers besides 42 can be used too.

I used Sections 3.1 and 3.2 of the NIST Secure Hash Standard to understand how SHA-256 represents and uses 32-bit words. I also looked at NumPy's documentation to confirm that `np.uint32` uses 32 bits and is unsigned.

### Sequences of 32-bit Words

In [3]:
words = np.array([42, 255, 1024], dtype=np.uint32)

print(words)
print(words.dtype)
print(len(words))

[  42  255 1024]
uint32
3


A sequence of 32-bit words is a collection of multiple 32-bit integers stored together. In Python, we can use a NumPy array to represent this sequence.

In the example, `np.array()` stores three numbers: 42, 255 and 1024. The `dtype=np.uint32` is responsible for making sure that each number is represented as an unsigned 32-bit integer. The `len()` function returns 3 because the array contains three elements. SHA-256 uses sequences of 32-bit words when processing messages, such as the sixteen 32-bit words that make up a 512-bit message block.

According to NIST FIPS 180-4, Section 3.1, a 512-bit message block in SHA-256 is represented as sixteen 32-bit words. This supports my decision to use a NumPy array to store multiple 32-bit words together.

### Input Messages

In [4]:
message = b"hello"

print(message)
print(type(message))
print(len(message))
print(list(message))

b'hello'
<class 'bytes'>
5
[104, 101, 108, 108, 111]


An input message is the data that we want SHA-256 to hash. Using a bytes object gives us a convenient way to access the individual bytes of our input message.

The code above demonstrates an example using the word `hello`, which contains 5 bytes, making it 40 bits altogether. The `type()` function shows that the message is a bytes object, while `len()` returns the number of bytes. The `list()` function displays the individual byte values.

According to NIST FIPS 180-4, Section 2.2.1, the symbol M represents the message to be hashed. In this implementation, I chose to use a Python bytes object to represent the input message because it allows the data to be accessed byte by byte.

### 512-bit Message Blocks

In [5]:
block = np.zeros(16, dtype=np.uint32)

print(block)
print(block.dtype)
print(len(block))
print(block.nbytes)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
uint32
16
64


A 512-bit message block in SHA-256 is a fixed-size chunk of data measuring 512 bits or 64 bytes.

We use `np.zeros(16, dtype=np.uint32)` to create an array that contains sixteen unsigned 32-bit integers, all initially set to 0.

The `len(block)` function returns 16 because the array contains sixteen elements, while `block.nbytes` returns 64 because the elements occupy 64 bytes, equivalent to 512 bits.

NIST FIPS 180-4, Section 3.1 tells us that a 512-bit message block is represented as sixteen 32-bit words. In this implementation, I use a NumPy array containing sixteen `np.uint32` elements to represent the structure of a message block.

### Final 256-bit Hash Value

In [6]:
hash_value = np.zeros(8, dtype=np.uint32)

print(hash_value)
print(hash_value.dtype)
print(len(hash_value))
print(hash_value.nbytes)

[0 0 0 0 0 0 0 0]
uint32
8
32


The final SHA-256 hash value contains 256 bits. This value can be represented using 8 32-bit words.

In the Python example, I used `np.zeros()` to create an array of 8 32-bit unsigned integers, which are all initially set to zero. The variable `hash_value` stores the array created by `np.zeros()`. The zeros are only placeholders to demonstrate how the final hash can be represented rather than an actual calculated SHA-256 hash.

The first print statement displays the eight zeros, while `hash_value.dtype` displays the data type of the elements, which is `uint32`. The `len()` function shows that the array contains 8 elements and `hash_value.nbytes` shows that they occupy 32 bytes in total.

### Big-endian Integers

In [7]:
number = 0x12345678

big_endian = number.to_bytes(4, byteorder="big")
little_endian = number.to_bytes(4, byteorder="little")

print(big_endian.hex())
print(little_endian.hex())

12345678
78563412


Big-endian describes the order in which bytes are stored when representing a number. SHA-256 uses `byteorder="big"` because it follows big-endian byte order meaning the most significant byte comes first. The difference between big-endian and little-endian is that in big-endian the most significant byte comes first, while in little-endian the least significant byte comes first.

The variable `number` stores a hexadecimal value, `0x12345678`, which is used to demonstrate the difference between big-endian and little-endian byte order.

The `to_bytes()` method converts the number into 4 bytes (32 bits). The `byteorder="big"` argument specifies that the most significant byte comes first. The `little_endian` variable is included to demonstrate the difference between big-endian and little-endian byte order.

The first print statement displays `12345678` because it uses big-endian byte order. The most significant byte is `12`, which is why it appears first. The second print statement displays `78563412` because it represents little-endian byte order where the least significant byte comes first, which in this case is `78`.

NIST FIPS 180-4, Section 3.1 describes how bit strings and integers are represented in SHA-256 using big-endian order.

**References**

- [NIST FIPS 180-4: Secure Hash Standard](https://doi.org/10.6028/NIST.FIPS.180-4) — Sections 3.1, 3.2 and 6.2
- [NumPy Scalar Types Documentation](https://numpy.org/doc/stable/reference/arrays.scalars.html) — `numpy.uint32`.

## Problem 2: SHA-256 Bitwise Operations

### Choose Function (Ch)

The `Ch` function takes three 32-bit unsigned integers, `x`, `y`, and `z` and works on them one bit position at a time.

At each position, the bit of `x` acts as the chooser: if it is `1`, the result takes the bit from `y` and if it is `0`, the result takes the bit from `z`. The function returns a single 32-bit unsigned integer containing the selected bits. 

Section 4.1.2 of NIST FIPS 180-4 defines the Choose function using logical operators. In Python, these can be represented using & (AND), ^ (XOR), and ~ (NOT).

`(x & y) ^ (~x & z)`

The `(x & y)` operation returns `1` only when both corresponding bits in `x` and `y` are `1`.

The `(~x & z)` operation returns `1` when the bit in `x` is `0` and the corresponding bit in `z` is `1`. This is because NOT flips the bit in `x` from `0` to `1`, and AND returns `1` when both bits are `1`.

The XOR (`^`) operator combines the two results and returns `1` when the corresponding bits are different. If the bits have the same value, the result is `0`.

When a bit in `x` is `1`, the first part `(x & y)` gives us the corresponding bit from `y`, while the second part `(~x & z)` gives `0`. After XOR, the result is the bit from `y`.

When a bit in `x` is `0`, the first part `(x & y)` gives `0`, while the second part `(~x & z)` gives us the corresponding bit from `z`. After XOR, the result is the bit from `z`.

#### Worked Example

This example uses four-bit values. The real function does exactly the same thing on 32 bits.

```text
x = 1100
y = 1010
z = 0110
```

1. `x & y` gives `1000`, because AND returns `1` only where both bits are `1`.
2. Flipping the four bits of `x` gives `~x = 0011`, and `~x & z` gives `0010`.
3. XOR combines `1000` and `0010` to give `1010` (10 in decimal).

This matches the "choose" rule. Where `x` is `1`, the result copies `y`, and where `x` is `0`, it copies `z`.

In [8]:
def Ch(x, y, z):
    """
    Apply the SHA-256 Choose function to three 32-bit unsigned integers.

    Parameters:
        x: The 32-bit unsigned integer used to select bits.
        y: The 32-bit unsigned integer selected when a bit in x is 1.
        z: The 32-bit unsigned integer selected when a bit in x is 0.

    Returns:
        A 32-bit unsigned integer containing the selected bits.
    """
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    return (x & y) ^ (~x & z)

In [9]:
result = Ch(0b1100, 0b1010, 0b0110)
print(result)

10


In [10]:
# x chooses bits from y (where x is 1) and z (where x is 0)
assert Ch(0b1100, 0b1010, 0b0110) == 0b1010

# x all zeros returns z
assert Ch(0b0000, 0b1010, 0b0110) == 0b0110
assert Ch(0, 0x12345678, 0xABCDEF01) == 0xABCDEF01

# x all ones returns y
assert Ch(0b1111, 0b1010, 0b0110) == 0b1010
assert Ch(0xFFFFFFFF, 0x12345678, 0xABCDEF01) == 0x12345678

### Majority Function (Maj)

The `Maj` function takes three 32-bit unsigned integers, `x`, `y` and `z`, and compares the bits in the same position. At each bit position, if at least two of the three bits are `1`, the function returns `1` for that position. Otherwise, it returns `0`. The function produces a single 32-bit unsigned integer containing the majority bit from each position.

Section 4.1.2 of NIST FIPS 180-4 defines the Majority function using AND and XOR operations. In Python, the formula can be written as:

`(x & y) ^ (x & z) ^ (y & z)`

The three AND operations compare each possible pair of bits: `x` and `y`, `x` and `z`, and `y` and `z`. If fewer than two of the input bits are `1`, all three AND operations return `0`, so the result is `0`.

When exactly two input bits are `1`, exactly one AND operation returns `1` and the other two return `0`, so XOR gives `1`.

When all three input bits are `1`, each AND operation returns `1`. XOR then combines these results as `1 ^ 1 ^ 1`, which gives `1`.

#### Example

This example uses four-bit values to demonstrate the Majority function. The actual SHA-256 function works with 32-bit unsigned integers.

```text
x = 1100
y = 1010
z = 1110
```

At the first bit position, all three bits are `1`, so the result is `1`. At the second and third positions, two of the three bits are `1`, so both positions also return `1`. At the final position, all three bits are `0`, so the result is `0`.

Therefore, the Majority function returns `1110` (14 in decimal).

In [11]:
def Maj(x, y, z):
    """
    Apply the SHA-256 Majority function to three 32-bit unsigned integers.

    Parameters:
        x: First 32-bit unsigned integer.
        y: Second 32-bit unsigned integer.
        z: Third 32-bit unsigned integer.

    Returns:
        A 32-bit unsigned integer containing the majority bits.
    """
    x = np.uint32(x)
    y = np.uint32(y)
    z = np.uint32(z)

    return (x & y) ^ (x & z) ^ (y & z)

In [ ]:
# Example from the explanation above
assert Maj(0b1100, 0b1010, 0b1110) == 0b1110

# Inputs are the same, the majority is that same value
assert Maj(0, 0, 0) == 0
assert Maj(0xFFFFFFFF, 0xFFFFFFFF, 0xFFFFFFFF) == 0xFFFFFFFF

# One input all ones and one all zeros: the third input decides every bit
assert Maj(0xFFFFFFFF, 0x00000000, 0x12345678) == 0x12345678

## Problem 3: Generating the SHA-256 Constants

## Problem 4: Padding and Parsing Messages

## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256

# End